# POIデータから地域特徴量を作る

保存済みOvertureを使い、吉祥寺の矩形AOI内の場所ごとの登録密度とカテゴリ構成を調べます。新しいデータは取得しません。

実測では3,666件を250mの25区画・500mの9区画へ割り当てました。店舗の営業状況・地域の優劣・他の街との類似性は評価しません。

[方法と出典](../docs/region-features.md) / [検証結果](../docs/region-features-validation.md) / [記事引き継ぎ](../docs/region-features-article-handoff.md)

## 1. 準備と固定条件

`projects/poi/` またはその `notebooks/` から実行します。既存の `osm`・`notebooks` 依存グループを使います。

- 入力はconfigs/region_features.tomlの実行IDとSHA-256で固定
- 第IX系 EPSG:6677、東・北の座標、共通原点(0,0)
- 格子の内部境界は東側・北側に一意割当、AOI外周は包含
- 密度の分母はAOIと交差する実面積
- 構成比の分母はカテゴリ欠損も含む全登録件数。営業状態による除外なし
- taxonomyの別名・階層は重複計上せず、単一basic_categoryを使用
- 0件区画の構成比は未定義。未取得データは0件とせず停止

In [ ]:
from geoai_open_lab.region_features import load_inputs, run_experiment
from geoai_open_lab.region_figures import save_figures
from IPython.display import SVG, display

places, aoi, config, provenance = load_inputs()
print({"release": config["release"], "run": config["input_run"],
       "rows": len(places), "sha256": config["input_sha256"]})

## 2. 250m・500m格子で集計

入力版・AOI・行数・重複IDを検査してから、全区画を集計します。端の格子を含めて合計と面積を照合します。保存物はGit管理外data/cache/region_features/kichijoji-v1に置きます。

In [ ]:
results, manifest, output_dir = run_experiment()
manifest["summary"]

## 3. 密度を地図で確認

2つの解像度を同じ色尺度で比較します。背景地図は使わず、実際の投影座標上の区画を描きます。数字は件数、色は件/km²です。

In [ ]:
figures = save_figures(results, output_dir)
display(SVG(filename=str(figures[0])))

250mで最多の区画は707件、密度11,312件/km²です。500mでは区画をまとめるため、集中の見え方が変わります。面積が小さい端の区画は率が変動しやすく、色だけで特徴を断定しません。

## 4. カテゴリ構成を見る

全25区画を北から南、同じ行は西から東へ並べます。表示用対応表は設定ファイルにあり、「その他」と「カテゴリ欠損」は別です。

In [ ]:
display(SVG(filename=str(figures[1])))

構成比は規模を表しません。同じ構成比でも件数が違うため、各区画のnと面積を併記しています。元のbasic_category別の表も保存済みです。

## 5. 欠損と営業状態を確認

欠損は0件でも営業中でもありません。

In [ ]:
print("カテゴリ欠損:", manifest["category_counts"].get("__MISSING__", 0))
print("営業状態:", manifest["status_counts"])
print("対象面積 (km²):", manifest["area_m2"] / 1_000_000)
print("250mと500mの合計:", {s: int(c.poi_count.sum()) for s, (c, _) in results.items()})

## まとめ

- 同一データでも、件数・密度・構成比は異なる側面を表します。
- カテゴリ欠損55件、営業状態欠損3,665件。営業中店舗数とは言えません。
- 今回0件区画はありませんが、実装は0件と未取得・カテゴリ欠損を区別します。
- 吉祥寺の実験用矩形内部だけの結果です。駅の縄張り・実際の商圏ではありません。
- 街同士の比較・普遍的地域分類・クラスタリング・Embeddingは後続実験です。

公開用Notebookには出力を残しません。実行済み版・HTML・図表はdata内へ保存してください。